<a href="https://colab.research.google.com/github/juliaR0/SF_10_new/blob/main/notebooks/S_Prototype_2_2_112526.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prototype 2.2 — Meaningful: measurement tools (no new training)

*Code in this notebook was drafted with Claude (Anthropic) and run, debugged and adapted by me. Prompts are in my AI log.*

**Goal:** build the tools that make outputs mean something, and test them on the runs I already have from 2.1 (healthy twins, damage sweep, therapy). Nothing here trains a model (except optional Tool 6).

| Tool | Question | Priority it supports |
|---|---|---|
| 1. Error categories | Is a failure switching, structure, tense/aux, article, or a wrong word? | #1 agrammatism |
| 2. Tense scorer | Right tense/auxiliary? (Chen's #1 error) | #1 agrammatism |
| 3. Confidence intervals | How sure are we about an average over twins? | all |
| 4. Sham-adjusted gain (paired) | Does therapy beat just retraining? | #2 transfer |
| 5. Twins needed | How many twins does 3c need? | #3 enough twins |
| 6. (optional) Damage 0.7 / 0.8 | Does pred_compress damage reach "moderate"? | #1, plans 3b |

**Why no new training:** toy language 2.0 and a Khoe-style twin population come in prototype 3 and mean retraining anyway. These tools carry over.

**Run order:** Section 1, then any section. Every rule is marked `# RULE` and gets a decision-log row.

# 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/dual_path

Mounted at /content/drive
/content/drive/MyDrive/dual_path


In [ ]:
import sys, os, glob, math, copy, pickle, shutil, subprocess
import concurrent.futures as cf
import pandas as pd, numpy as np
from scipy import stats
sys.path.insert(0, 'src')
pd.set_option('display.max_rows', 200, 'display.max_colwidth', 100)

def lang_of(msg):
    return 'es' if 'TARGET-LANG=es' in str(msg) else 'en'

def structure(msg):
    msg = str(msg)
    if 'RECIPIENT=' in msg: return '3 ditransitive'
    if 'PATIENT=' in msg:   return '2 transitive'
    if 'ATTR=' in msg:      return 'copular'
    return '1 intransitive'

LEX = pd.read_csv('data/code-switching/lexicon.csv')
ES_ONLY = set(LEX.morpheme_es.dropna()) - set(LEX.morpheme_en.dropna())
EN_ONLY = set(LEX.morpheme_en.dropna()) - set(LEX.morpheme_es.dropna())

def load_test(path, epoch=None):
    d = pd.read_csv(path)
    if epoch is not None: d = d[d.epoch == epoch].copy()
    d['m'] = d['meaning'].astype(str).str.strip()
    d['tlang'] = d.message.apply(lang_of)
    d['struct'] = d.message.apply(structure)
    def lang_ok(r):
        words = set(str(r.produced_sentence).split())
        wrong = ES_ONLY if r.tlang == 'en' else EN_ONLY
        return not (words & wrong)
    d['lang_ok'] = d.apply(lang_ok, axis=1)
    d['strict'] = (d.m == 'True') & d.lang_ok
    return d

def pct(mask): return round(100 * mask.mean(), 1)

**Where 2.1's files are.** The damage chosen per twin in 2.1 (from its calibration table) is copied here.

In [ ]:
PATIENTS = ['P14', 'P77']
N_TWINS = 3
FINAL_EPOCH = 15
SESSIONS = 5
OPTIONS = ['EN', 'ES', 'MIXED', 'SHAM']
CHOSEN_DAMAGE = {('P14', 1): 0.6, ('P14', 2): 0.6, ('P14', 3): 0.5,
                 ('P77', 1): 0.5, ('P77', 2): 0.6, ('P77', 3): 0.6}   # from 2.1

def healthy_path(pid, k):   return f'simulations/p21_{pid}_twins/{k}/test.csv'
def damaged_path(pid, k, f): return f'simulations/p21_{pid}_t{k}_les{int(round(f*100)):02d}_eval/1/test.csv'
def therapy_path(pid, k, o): return f'simulations/p21_{pid}_t{k}_th{o}_run/1/test.csv'

missing = [p for pid in PATIENTS for k in range(1, N_TWINS + 1)
           for p in [healthy_path(pid, k), damaged_path(pid, k, CHOSEN_DAMAGE[(pid, k)])] + [therapy_path(pid, k, o) for o in OPTIONS]
           if not os.path.exists(p)]
print('missing files:', len(missing)); print(*missing[:10], sep='\n')
print('2.1 outcomes table present:', os.path.exists('analysis/p21_therapy_outcomes.csv'))

missing files: 0

2.1 outcomes table present: True


# 2. Tools 1 + 2 — Error categories and tense scorer

Every answer gets **one** category, checked in this order:

| Category | Meaning | Example (target: *the man is carrying the ball*) |
|---|---|---|
| correct | strict score passes | *the man is carrying the ball* |
| switch | right message, but other-language words | *the man is carrying la pelota* |
| structure | wrong skeleton: a missing, extra or reordered part | *the man is carrying* |
| tense/aux | skeleton right, verb tense or auxiliary wrong | *the man carries the ball* · *the man carrying the ball* |
| article | only the determiners differ | *a man is carrying the ball* |
| wrong word | skeleton, tense, articles right, but a content word is wrong (anomic-like) | *the woman is carrying the ball* |
| other | anything else | |

**How:** each word is looked up in `lexicon.csv` (in either language) to get its part of speech, tense and aspect.
- **Skeleton** = the content-word parts of speech in order (noun, verb, adjective, preposition, pronoun), without articles and auxiliaries.
- **Tense signature** = the verb's tense/aspect plus which auxiliaries appear.

`# RULE` Category order and definitions above. `# RULE` Auxiliaries (*is, has, está, ha*) count as tense markers, not skeleton. `# RULE` Only answers whose language is right can be structure / tense / article / wrong-word errors; wrong-language answers with the right message are "switch"; wrong-language answers with the wrong message fall in the other categories too, and `lang_ok` is kept as its own column.

def category(r):
    if r.strict: return 'correct'
    if r.m == 'True' and not r.lang_ok: return 'switch'
    ps, pt, pd_, pc = analyse(r.produced_sentence)
    ts, tt, td, tc = analyse(r.target_sentence)
    if ps != ts: return 'structure'
    if pt != tt: return 'tense/aux'
    if pd_ != td: return 'article'
    if pc != tc: return 'wrong word'
    return 'other'

CATS = ['correct', 'switch', 'structure', 'tense/aux', 'article', 'wrong word', 'other']


In [ ]:
# word -> (pos, tense_or_aspect, concept), both languages
WORD = {}
for r in LEX.itertuples():
    pos = str(r.pos).strip()
    ta = ':'.join(str(x) for x in [r.tense, r.aspect] if isinstance(x, str))
    for w in [r.morpheme_es, r.morpheme_en]:
        if isinstance(w, str):
            WORD.setdefault(w.strip(), (pos, ta, str(r.concept)))
# RULE: a word that is ever an auxiliary (is, está, has, ha) is treated as a tense marker everywhere, copula included
for w in set(LEX[LEX.pos == 'aux'].morpheme_es.dropna()) | set(LEX[LEX.pos == 'aux'].morpheme_en.dropna()):
    WORD[w.strip()] = ('aux', 'present', 'AUX')

CONTENT = {'noun', 'verb', 'participle', 'adj', 'prep', 'pron'}

def analyse(sentence):
    toks = [t for t in str(sentence).split() if t != '.']
    info = [WORD.get(t, ('?', '', t)) for t in toks]
    skeleton = tuple('verb' if p == 'participle' else p for p, _, _ in info if p in CONTENT)
    tense = tuple((p, ta) for (p, ta, _) in info if p in ('verb', 'participle', 'aux'))
    dets = tuple(t for t, (p, _, _) in zip(toks, info) if p == 'det')
    concepts = tuple(c for (p, _, c) in info if p in ('noun', 'verb', 'participle', 'adj'))
    return skeleton, tense, dets, concepts

def category(r):
    if r.strict: return 'correct'
    ps, pt, pd_, pc = analyse(r.produced_sentence)
    ts, tt, td, tc = analyse(r.target_sentence)
    # RULE: wrong-language words + right message (or same skeleton and same meanings) = switch
    if not r.lang_ok and (r.m == 'True' or (ps == ts and pc == tc)): return 'switch'
    # RULE: a repeated content word (perseveration, e.g. "the book libro") = structure error
    repeats = len(set(pc)) < len(pc) and len(set(tc)) == len(tc)
    if ps != ts or repeats: return 'structure'
    if pt != tt: return 'tense/aux'
    if pc != tc: return 'wrong word'
    if pd_ != td: return 'article'
    return 'agreement'   # same words and meanings, different form (e.g. tranquilo vs tranquila)

CATS = ['correct', 'switch', 'structure', 'tense/aux', 'wrong word', 'article', 'agreement']
def categorise(d):
    d = d.copy(); d['cat'] = d.apply(category, axis=1); return d

def cat_table(d, by):
    t = pd.crosstab([d[b] for b in by], d.cat, normalize='index').mul(100).round(1)
    return t.reindex(columns=[c for c in CATS if c in t.columns])

**Sanity check first:** run the categories on a handful of real answers and read them. If a category looks wrong, fix `analyse()` before trusting any table.

In [ ]:
d = categorise(load_test(damaged_path('P77', 1, CHOSEN_DAMAGE[('P77', 1)]), epoch=0))
d[d.cat != 'correct'].groupby('cat').head(3)[['cat', 'target_sentence', 'produced_sentence']]

,cat,target_sentence,produced_sentence
1,wrong word,el anfitrión cansado está empujando una silla .,el anfitrión tranquilo está empujando una silla .
4,switch,she is pushing a chair .,she is pushing una silla .
5,switch,he has the toy .,él tiene el juguete .
7,structure,él está amable .,él llega .
8,switch,the brother dies .,the brother muere .
10,structure,she is carrying a chair .,she is llevando .
11,wrong word,el perro inteligente corrió .,el perro inteligente swam .
12,structure,él muestra a_ una abuela la silla .,él muestra a_ una abuela .
18,wrong word,el abuelo cansado ha empujado el bolígrafo .,el abuelo tranquilo ha empujado el bolígrafo .
23,tense/aux,una abuela está corriendo .,una abuela corre .


## 2a. Healthy vs damaged: what does the damage do?

Preview of 3b. If the damage is agrammatism-like, **structure** and **tense/aux** should grow more than **wrong word**. If **switch** grows most, the damage mostly breaks language choice.

In [ ]:
rows = []
for pid in PATIENTS:
    for k in range(1, N_TWINS + 1):
        h = load_test(healthy_path(pid, k), epoch=FINAL_EPOCH); h['state'] = '1 healthy'
        dm = load_test(damaged_path(pid, k, CHOSEN_DAMAGE[(pid, k)]), epoch=0); dm['state'] = '2 damaged'
        for x in (h, dm):
            x['patient'], x['twin'] = pid, k
            rows.append(categorise(x))
HD = pd.concat(rows)
HD.to_csv('analysis/p22_healthy_damaged_categories.csv', index=False)
cat_table(HD, ['patient', 'tlang', 'state'])

cat                      correct  switch  structure  tense/aux  wrong word  \
patient tlang state                                                          
P14     en    1 healthy      6.0     4.0       37.3        2.7        48.0   
              2 damaged      0.0    18.7       64.7        1.3        14.7   
        es    1 healthy     98.0     0.4        1.1        0.0         0.4   
              2 damaged     68.7     3.7       19.6        6.9         0.3   
P77     en    1 healthy     61.5    30.0        5.0        0.2         2.7   
              2 damaged     50.5    23.1       21.1        3.9         1.1   
        es    1 healthy     82.6     3.9        3.8        0.0         9.1   
              2 damaged     43.3    21.1       23.0        6.2         4.1   

cat                      article  agreement  
patient tlang state                          
P14     en    1 healthy      0.0        2.0  
              2 damaged      0.0        0.7  
        es    1 healthy      0.0        0.1  
              2 damaged      0.0        0.7  
P77     en    1 healthy      0.5        0.0  
              2 damaged      0.2        0.1  
        es    1 healthy      0.0        0.6  
              2 damaged      1.4        0.9

## 2b. Damage curve: categories by damage level (all 2.1 calibration runs)

In [ ]:
rows = []
for pid in PATIENTS:
    for k in range(1, N_TWINS + 1):
        for p in sorted(glob.glob(f'simulations/p21_{pid}_t{k}_les*_eval/1/test.csv')):
            lvl = int(p.split('_les')[1][:2]) / 100
            x = categorise(load_test(p, epoch=0)); x['patient'], x['twin'], x['damage'] = pid, k, lvl
            rows.append(x)
CURVE = pd.concat(rows)
cat_table(CURVE, ['patient', 'damage'])

cat             correct  switch  structure  tense/aux  wrong word  article  \
patient damage                                                               
P14     0.0        88.8     0.7        4.7        0.3         5.2      0.0   
        0.2        86.4     1.3        6.7        0.7         4.6      0.0   
        0.3        85.1     1.3        8.3        1.3         3.9      0.0   
        0.4        82.5     1.8       10.5        1.5         3.5      0.0   
        0.5        71.1     4.1       18.1        3.1         3.2      0.0   
        0.6        55.0     4.9       30.7        7.1         1.5      0.0   
        0.7        47.4     4.9       39.1        7.1         0.6      0.0   
        0.8        24.9     6.1       60.4        6.5         0.5      0.3   
P77     0.0        70.8    18.5        4.5        0.1         5.5      0.3   
        0.2        67.5    18.4        8.5        0.6         3.9      0.3   
        0.3        67.1    18.2       10.2        0.7         2.9      0.4   
        0.4        61.7    20.2       12.1        2.3         2.8      0.1   
        0.5        56.4    18.3       17.5        4.2         2.4      0.6   
        0.6        44.7    23.5       24.7        4.3         1.7      0.7   
        0.7        32.2    24.8       35.9        5.2         1.7      0.1   
        0.8        21.5    15.8       55.3        5.3         1.3      0.5   

cat             agreement  
patient damage             
P14     0.0           0.3  
        0.2           0.3  
        0.3           0.2  
        0.4           0.2  
        0.5           0.3  
        0.6           0.8  
        0.7           0.9  
        0.8           1.4  
P77     0.0           0.3  
        0.2           0.7  
        0.3           0.5  
        0.4           0.7  
        0.5           0.6  
        0.6           0.4  
        0.7           0.1  
        0.8           0.3

## 2c. Tense/aux errors by structure (damaged twins)

Chen's top error was wrong verb tense. Which structures get the tense errors?

In [ ]:
dm = HD[HD.state == '2 damaged']
(dm.assign(tense_err = dm.cat == 'tense/aux')
   .groupby(['patient', 'tlang', 'struct']).tense_err.mean().mul(100).round(1).unstack())

struct         1 intransitive  2 transitive  3 ditransitive  copular
patient tlang                                                       
P14     en                0.0           1.6             0.0      7.7
        es               10.2           6.8             0.0      2.3
P77     en                8.9           1.3             0.0      1.3
        es               11.4           4.2             0.0      1.6

# 3. Therapy re-scored with the categories

Same therapy runs as 2.1, now asking: after therapy, did **structure** errors go down (real grammar recovery) or did **switch** errors go down (language-choice recovery)? Ditransitives only, before (epoch 0) vs after (epoch 5).

In [ ]:
rows = []
for pid in PATIENTS:
    for k in range(1, N_TWINS + 1):
        for o in OPTIONS:
            for ep, when in [(0, 'before'), (SESSIONS, 'after')]:
                x = load_test(therapy_path(pid, k, o), epoch=ep)
                x = categorise(x[x.struct == '3 ditransitive'])
                x['patient'], x['twin'], x['option'], x['when'] = pid, k, o, when
                rows.append(x)
TH = pd.concat(rows)
t = cat_table(TH, ['patient', 'option', 'tlang', 'when'])
t

cat                          correct  switch  structure  wrong word  article
patient option tlang when                                                   
P14     EN     en    after       0.0     0.0       57.1        42.9      0.0
                     before      0.0     4.8       90.5         4.8      0.0
               es    after      23.8     7.0       60.1         9.1      0.0
                     before     36.4     2.8       60.8         0.0      0.0
        ES     en    after       0.0    38.1       57.1         4.8      0.0
                     before      0.0     4.8       90.5         4.8      0.0
               es    after      90.9     0.0        9.1         0.0      0.0
                     before     37.1     1.4       61.5         0.0      0.0
        MIXED  en    after       0.0     4.8       85.7         9.5      0.0
                     before      0.0     4.8       90.5         4.8      0.0
               es    after      80.4     0.0       19.6         0.0      0.0
                     before     37.1     2.8       60.1         0.0      0.0
        SHAM   en    after       0.0     4.8       95.2         0.0      0.0
                     before      0.0     4.8       90.5         4.8      0.0
               es    after      51.0     0.0       48.3         0.7      0.0
                     before     29.4     0.7       69.9         0.0      0.0
P77     EN     en    after      63.2    18.4       13.8         0.0      4.6
                     before     17.2     4.6       74.7         1.1      2.3
               es    after       0.0    59.4       40.6         0.0      0.0
                     before      3.1    18.8       78.1         0.0      0.0
        ES     en    after      10.3    36.8       44.8         4.6      3.4
                     before     17.2     4.6       74.7         1.1      2.3
               es    after      53.1    17.2       18.8         7.8      3.1
                     before      3.1    17.2       79.7         0.0      0.0
        MIXED  en    after      62.1    20.7       12.6         0.0      4.6
                     before     17.2     4.6       74.7         1.1      2.3
               es    after      39.1    37.5       23.4         0.0      0.0
                     before      3.1    20.3       76.6         0.0      0.0
        SHAM   en    after      33.3     9.2       57.5         0.0      0.0
                     before     17.2     4.6       74.7         1.1      2.3
               es    after      12.5    17.2       70.3         0.0      0.0
                     before      3.1    14.1       82.8         0.0      0.0

# 4. Tool 3 — Confidence intervals across twins

Each twin is one data point. 95% interval = mean ± t × (SD ÷ √n). With n = 3, t = 4.30, so intervals are wide.

`# RULE` t-interval across twins (not min/max). With more twins (prototype 3) switch to mixed-effects models (Khoe).

In [ ]:
def ci95(x):
    x = pd.Series(x).dropna(); n = len(x)
    if n < 2: return pd.Series({'mean': x.mean(), 'lo': np.nan, 'hi': np.nan, 'sd': np.nan, 'n': n})
    m, sd = x.mean(), x.std(ddof=1)
    h = stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n)
    return pd.Series({'mean': round(m, 1), 'lo': round(m - h, 1), 'hi': round(m + h, 1), 'sd': round(sd, 1), 'n': n})

res = pd.read_csv('analysis/p21_therapy_outcomes.csv')
res.groupby(['patient', 'option']).benefit.apply(ci95).unstack()

mean    lo     hi    sd    n
patient option                              
P14     EN     -13.9 -39.3   11.4  10.2  3.0
        ES      26.9  -8.0   61.8  14.0  3.0
        MIXED   21.3 -27.5   70.2  19.7  3.0
        SHAM    10.7  -5.8   27.3   6.7  3.0
P77     EN       9.6 -47.6   66.7  23.0  3.0
        ES       6.5 -34.0   47.0  16.3  3.0
        MIXED   38.7 -33.5  110.9  29.1  3.0
        SHAM    11.9 -30.6   54.4  17.1  3.0

# 5. Tool 4 — Sham-adjusted gain (paired within each twin)

For each twin: **option's gain − the same twin's sham gain**. Comparing a twin with itself removes twin-to-twin differences.

`# RULE` Main outcome = sham-adjusted gain, per language and as benefit. `# RULE` New recommend(): an option counts only if its sham-adjusted benefit interval is above 0; it's the clear winner only if its paired difference from the runner-up is above 0.

In [ ]:
wide = res.pivot_table(index=['patient', 'twin'], columns='option', values=['gain_en', 'gain_es', 'benefit'])
adj = []
for (pid, k), r in wide.iterrows():
    for o in ['EN', 'ES', 'MIXED']:
        adj.append({'patient': pid, 'twin': k, 'option': o,
                    'adj_gain_en': r[('gain_en', o)] - r[('gain_en', 'SHAM')],
                    'adj_gain_es': r[('gain_es', o)] - r[('gain_es', 'SHAM')],
                    'adj_benefit': r[('benefit', o)] - r[('benefit', 'SHAM')]})
ADJ = pd.DataFrame(adj)
ADJ.to_csv('analysis/p22_sham_adjusted.csv', index=False)
ADJ.groupby(['patient', 'option'])[['adj_gain_en', 'adj_gain_es', 'adj_benefit']].agg(lambda x: f"{ci95(x)['mean']} [{ci95(x)['lo']}, {ci95(x)['hi']}]")

adj_gain_en          adj_gain_es  \
patient option                                              
P14     EN            0.0 [0.0, 0.0]   -34.3 [-72.2, 3.5]   
        ES            0.0 [0.0, 0.0]  32.3 [-42.0, 106.6]   
        MIXED         0.0 [0.0, 0.0]  22.0 [-90.9, 134.9]   
P77     EN       30.5 [-80.8, 141.7]  -11.9 [-42.2, 18.3]   
        ES      -23.0 [-102.2, 56.3]    40.4 [28.7, 52.2]   
        MIXED    29.1 [-65.3, 123.5]   25.3 [-39.6, 90.3]   

                         adj_benefit  
patient option                        
P14     EN      -24.7 [-34.2, -15.2]  
        ES        16.2 [-21.0, 53.4]  
        MIXED     10.6 [-44.9, 66.1]  
P77     EN        -2.3 [-74.2, 69.6]  
        ES        -5.4 [-62.0, 51.3]  
        MIXED     26.8 [-38.4, 91.9]

In [ ]:
def recommend_v2(pid):
    a = ADJ[ADJ.patient == pid]
    summ = a.groupby('option').adj_benefit.apply(ci95).unstack().sort_values('mean', ascending=False)
    best, second = summ.index[0], summ.index[1]
    diff = (a[a.option == best].set_index('twin').adj_benefit - a[a.option == second].set_index('twin').adj_benefit)
    d = ci95(diff)
    if not summ.loc[best, 'lo'] > 0:
        verdict = f'no option clearly beats sham (best: {best})'
    elif d['lo'] > 0:
        verdict = f'{best}, clearly ahead of {second}'
    else:
        verdict = f'{best} beats sham, but not clearly ahead of {second}'
    return summ, d, verdict

for pid in PATIENTS:
    summ, d, verdict = recommend_v2(pid)
    print('=' * 60, '\n', pid); print(summ.to_string())
    print(f'best minus runner-up: {d["mean"]} [{d["lo"]}, {d["hi"]}]'); print('RECOMMENDATION:', verdict)

 P14
        mean    lo    hi    sd    n
option                             
ES      16.2 -21.0  53.4  15.0  3.0
MIXED   10.6 -44.9  66.1  22.4  3.0
EN     -24.7 -34.2 -15.2   3.8  3.0
best minus runner-up: 5.6 [-14.7, 25.9]
RECOMMENDATION: no option clearly beats sham (best: ES)
 P77
        mean    lo    hi    sd    n
option                             
MIXED   26.8 -38.4  91.9  26.2  3.0
EN      -2.3 -74.2  69.6  28.9  3.0
ES      -5.4 -62.0  51.3  22.8  3.0
best minus runner-up: 29.1 [-5.4, 63.6]
RECOMMENDATION: no option clearly beats sham (best: MIXED)


# 6. Tool 5 — How many twins does prototype 3c need?

To pin an average down to ± M points: n ≈ (2 × SD ÷ M)², using the twin-to-twin SD of the sham-adjusted gains.

`# RULE` Target precision M = 5 points (a therapy effect smaller than ~5 points is not clinically interesting in this toy setting). Revisit when toy language 2.0 changes the scale.

In [ ]:
M = 5                                                    # RULE
sd = ADJ.groupby(['patient', 'option'])[['adj_gain_en', 'adj_gain_es', 'adj_benefit']].std(ddof=1).round(1)
need = (2 * sd / M).pow(2).apply(np.ceil)
print('Twin-to-twin SD (points):'); display(sd)
print(f'Twins needed for ±{M} points:'); display(need)

Twin-to-twin SD (points):


adj_gain_en  adj_gain_es  adj_benefit
patient option                                       
P14     EN              0.0         15.2          3.8
        ES              0.0         29.9         15.0
        MIXED           0.0         45.4         22.4
P77     EN             44.8         12.2         28.9
        ES             31.9          4.7         22.8
        MIXED          38.0         26.1         26.2

Twins needed for ±5 points:


adj_gain_en  adj_gain_es  adj_benefit
patient option                                       
P14     EN              0.0         37.0          3.0
        ES              0.0        144.0         36.0
        MIXED           0.0        330.0         81.0
P77     EN            322.0         24.0        134.0
        ES            163.0          4.0         84.0
        MIXED         232.0        109.0        110.0

# 7. Tool 6 (optional, ~5 min) — Does damage 0.7 / 0.8 reach 60% of healthy?

Uses 2.1's healthy twins and lesion code. Skip if short on time; it only plans 3b.

In [ ]:
import lz4.frame, torch

def load_weights(path):
    with lz4.frame.open(path, 'rb') as f: return pickle.load(f)

def lesion(layers, layer_name, fraction, seed=0):
    d = copy.deepcopy(layers); W = d[layer_name].in_weights
    g = torch.Generator().manual_seed(seed)
    d[layer_name].in_weights = W.masked_fill(torch.rand(W.shape, generator=g) < fraction, 0.0)
    return d

def save_layers(layers, folder, epoch):
    for path in [f'{folder}/1/weights', f'{folder}/weights']:
        os.makedirs(path, exist_ok=True)
        with lz4.frame.open(f'{path}/w{epoch}.lz4', 'wb') as f: pickle.dump(layers, f, protocol=-1)

def build_input(folder, src_run, src_sim):
    os.makedirs(f'{folder}/input', exist_ok=True); os.makedirs(f'{folder}/1', exist_ok=True)
    for f in os.listdir(f'{src_run}/input'):
        p = f'{src_run}/input/{f}'
        if os.path.isfile(p): shutil.copyfile(p, f'{folder}/input/{f}')
    for f in ['training.in', 'test.in']: shutil.copyfile(f'{src_run}/{src_sim}/{f}', f'{folder}/1/{f}')
    return folder

WORKERS = max(1, os.cpu_count() - 2)
def run_many(jobs):
    todo = [(a, r) for a, r in jobs if not os.path.exists(f'simulations/{r}')]
    print(len(jobs) - len(todo), 'already done;', len(todo), 'to run on', WORKERS, 'workers')
    env = dict(os.environ, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
    def one(job):
        a, r = job
        p = subprocess.run(f'python src/start_dual_path.py {a} --resdir {r}', shell=True, capture_output=True, text=True, env=env)
        return r, p.returncode, p.stderr[-400:]
    with cf.ThreadPoolExecutor(WORKERS) as ex:
        for r, code_, err in ex.map(one, todo):
            print('ok    ' if code_ == 0 else 'FAILED', r, '' if code_ == 0 else err)

jobs = []
for pid in PATIENTS:
    run = f'simulations/p21_{pid}_twins'
    for k in range(1, N_TWINS + 1):
        w = load_weights(f'{run}/{k}/weights/w{FINAL_EPOCH}.lz4')
        args = f"--norandomization --sim 1 --hidden {w['hidden'].size} --compress {w['compress'].size} --fw 15 --cs --lang es en"
        for frac in [0.7, 0.8]:
            tag = f'p21_{pid}_t{k}_les{int(frac*100):02d}'
            folder = f'simulations/{tag}_w'
            if not os.path.exists(f'{folder}/weights/w{FINAL_EPOCH}.lz4'):
                save_layers(lesion(w, 'pred_compress', frac, seed=k), folder, FINAL_EPOCH)
            inp = build_input(f'simulations/{tag}_in', run, k)
            jobs.append((f'{args} --epochs 1 --sw {folder} --swe {FINAL_EPOCH} --input {inp}', f'{tag}_eval'))
run_many(jobs)

0 already done; 12 to run on 1 workers
ok     p21_P14_t1_les70_eval 
ok     p21_P14_t1_les80_eval 
ok     p21_P14_t2_les70_eval 
ok     p21_P14_t2_les80_eval 
ok     p21_P14_t3_les70_eval 
ok     p21_P14_t3_les80_eval 
ok     p21_P77_t1_les70_eval 
ok     p21_P77_t1_les80_eval 
ok     p21_P77_t2_les70_eval 
ok     p21_P77_t2_les80_eval 
ok     p21_P77_t3_les70_eval 
ok     p21_P77_t3_les80_eval 


In [ ]:
rows = []
for pid in PATIENTS:
    for k in range(1, N_TWINS + 1):
        h = pct(load_test(damaged_path(pid, k, 0.0), epoch=0).strict)
        for f in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
            p = damaged_path(pid, k, f)
            if os.path.exists(p):
                rows.append({'patient': pid, 'twin': k, 'damage': f, 'ratio': round(pct(load_test(p, epoch=0).strict) / h, 2)})
pd.DataFrame(rows).pivot_table(index=['patient', 'twin'], columns='damage', values='ratio')

damage        0.0   0.2   0.3   0.4   0.5   0.6   0.7   0.8
patient twin                                               
P14     1     1.0  1.01  1.01  0.99  0.94  0.80  0.74  0.41
        2     1.0  0.95  0.92  0.89  0.77  0.59  0.56  0.12
        3     1.0  0.96  0.94  0.91  0.70  0.47  0.30  0.31
P77     1     1.0  0.89  0.85  0.70  0.66  0.52  0.37  0.13
        2     1.0  0.97  1.03  0.93  0.87  0.70  0.38  0.32
        3     1.0  0.98  0.95  0.94  0.83  0.65  0.58  0.42

**Real 0.0 check:** the damage-0.0 run should match the healthy check at epoch 15 (same weights, same test). Big differences mean the loading route changes something.

In [ ]:
pd.DataFrame([{'patient': pid, 'twin': k,
               'healthy_ep15_%': pct(load_test(healthy_path(pid, k), epoch=FINAL_EPOCH).strict),
               'damage0_%': pct(load_test(damaged_path(pid, k, 0.0), epoch=0).strict)}
              for pid in PATIENTS for k in range(1, N_TWINS + 1)])

,patient,twin,healthy_ep15_%,damage0_%
0,P14,1,87.8,87.8
1,P14,2,89.4,89.4
2,P14,3,89.2,89.2
3,P77,1,57.2,57.2
4,P77,2,73.0,73.0
5,P77,3,82.2,82.2


# 8. Prototype 2.2 summary *(fill in after running)*

**Error categories (healthy → damaged):**

**Is the damage agrammatism-like so far? (structure + tense/aux vs wrong word vs switch):**

**Therapy re-scored: what actually recovered?**

**Sham-adjusted results + new recommendation:**

**Twins needed for 3c:**

**Damage 0.7/0.8 + real 0.0 check:**

**Decision log rows added:** category order and definitions · auxiliaries as tense markers · t-interval across twins · sham-adjusted gain as main outcome · recommend_v2 rule · precision target M = 5

**Next (3a):** toy language 2.0 + NAVS-style battery, Khoe-style twin population.